# 4. Bericht: Schwarzfahrer im Schweizer ÖV

Gruppe 05 (SC02_Gruppe_05): Dominik Erbsland, Joel Liechti. ZHAW, CAS Information Engineering, Scripting.

## Einleitung

Schwarzfahren kostet die Schweizer ÖV-Unternehmen jedes Jahr Einnahmen. Kontrollen sind teuer, und das Personal kann nicht überall gleichzeitig sein. Wer weiss, wo und wann besonders viele Schwarzfahrer angetroffen werden, kann Kontrollen gezielter planen.

**Fragestellung:** An welchen Haltestellen, auf welchen Strecken und zu welchen Zeiten werden besonders viele Schwarzfahrer erfasst? Und wer wird erfasst?

**Wer interessiert sich dafür?**
- Verkehrsunternehmen wie SBB oder PostAuto, die ihre Kontrollen planen.
- Verkehrsverbünde wie der ZVV, die Einnahmen und Tarife verantworten.
- Leitungen des Kontrollpersonals für die Einsatzplanung.

## Daten

| Datensatz | Inhalt | Quelle |
|---|---|---|
| Kontrolldaten (`ASP_202509.XLSX`) | rund 86'700 Fälle mit Unternehmen, Zeit, Linie, Start, Ziel, Geschlecht, Geburtsdatum und Grund | eigener Datensatz, Export September 2025 |
| Haltestelle / Haltekante | alle Haltestellen der Schweiz mit Koordinaten | [opendata.swiss](https://opendata.swiss/en/dataset/haltestelle-haltekante) (SBB) |
| Kartenhintergrund | Strassenkarte für die Heatmap | [OpenStreetMap](https://www.openstreetmap.org) |

Die Kontrolldaten enthalten Geburtsdaten. Sie sind deshalb nicht im Repository. Für die Analyse verwenden wir nur Jahrgang und Altersgruppe.

## Vorgehen

1. **Daten laden und bereinigen** (Notebook 1): Duplikate entfernen, Zeit aufteilen, Gründe in Kategorien zusammenfassen, Unternehmensnummern durch Namen ersetzen und die Haltestellen über den Namen mit Koordinaten verbinden. Rund 98 % der Starthaltestellen werden gefunden.
2. **Analysieren** (Notebook 2): Kennzahlen nach Geschlecht, Alter, Zeit, Unternehmen, Grund, Haltestelle und Strecke berechnen und als CSV speichern.
3. **Darstellen** (Notebook 3): Überblicksdiagramm, Wochentag und Stunde als Raster, Jahrgänge und eine Heatmap auf der Schweizer Karte.

## Ergebnisse

Die folgenden Zahlen kommen direkt aus den Dateien von Notebook 2. Notebook 1 und 2 müssen vorher gelaufen sein.

In [ ]:
from pathlib import Path

import pandas as pd

PROCESSED = Path("../data/processed")

weekday_hour = pd.read_csv(PROCESSED / "faelle_wochentag_stunde.csv", index_col=0)
stations = pd.read_csv(PROCESSED / "faelle_haltestelle.csv")
companies = pd.read_csv(PROCESSED / "faelle_unternehmen.csv", index_col=0)["count"]
gender = pd.read_csv(PROCESSED / "faelle_geschlecht.csv", index_col=0)["count"]

print("Fälle insgesamt:", companies.sum())
print("Wochentag mit den meisten Fällen:", weekday_hour.sum(axis=1).idxmax())
print("Stunde mit den meisten Fällen:", weekday_hour.sum().idxmax(), "Uhr")
print("Haltestelle mit den meisten Fällen:", stations.iloc[0]["station_from"])
print("Unternehmen mit den meisten Fällen:", companies.index[0])
print("Anteil Geschlecht in %:")
print((gender / gender.sum() * 100).round(1))

**Wichtigste Erkenntnisse**

- **Ort:** Die meisten Fälle gibt es an den grossen Bahnhöfen, allen voran Zürich HB, danach Lausanne, Genf, Basel und Bern. Die Heatmap zeigt diese Zentren deutlich.
- **Zeit:** Am Wochenanfang gibt es mehr Fälle als am Wochenende. Über den Tag steigt die Zahl bis zum Abendverkehr um 17 Uhr.
- **Strecken:** Die häufigsten Strecken sind Zürich HB nach Basel SBB und Genève nach Lausanne, jeweils in beide Richtungen.
- **Unternehmen:** Die SBB hat mit Abstand am meisten Fälle, vermutlich auch, weil sie am meisten Fahrgäste befördert.
- **Grund:** Die meisten Personen haben gar kein gültiges Billett. Danach folgen Probleme mit E-Tickets und Check-In.
- **Personen:** Am häufigsten werden Personen zwischen 18 und 35 Jahren erfasst.

## Grenzen

- Wir sehen nur Fälle, bei denen jemand erwischt wurde. Wie oft ohne Befund kontrolliert wurde, wissen wir nicht. Viele Fälle an einem Ort können also auch einfach viele Kontrollen bedeuten.
- Der Code für das Geschlecht ist nicht dokumentiert. Die Zuordnung nach ISO 5218 ist eine Annahme und muss bestätigt werden.
- Einige Unternehmensnummern (zum Beispiel 490) konnten wir nicht sicher zuordnen.
- Viele Fälle sind um genau 0 Uhr erfasst. Das deutet auf fehlende Uhrzeiten hin.
- Rund 2 % der Haltestellen liegen im Ausland oder sind anders geschrieben und fehlen auf der Karte.

## Fazit

Kontrollen lohnen sich besonders an grossen Bahnhöfen, am Wochenanfang und im Abendverkehr. Für eine echte Schwarzfahrerquote bräuchte man zusätzlich die Anzahl aller Kontrollen oder die Fahrgastzahlen pro Haltestelle.